LangChain 常见 Loader 



### ① UnstructuredFileLoader —— 通用文件加载器

🔹 适用范围最广，可自动识别 txt / pdf / docx / html / md 等多种格式。
🔹 内部使用 unstructured 库，能自动提取正文、标题、表格、列表等结构化信息。

In [ ]:
from langchain.document_loaders import UnstructuredFileLoader

# 传入任意文件路径
loader = UnstructuredFileLoader("../data/example.txt")
docs = loader.load()

print(f"共加载 {len(docs)} 段文档")
print(docs)  # 打印前200字
print(docs[0].page_content[:200])  # 打印前200字


使用场景

	•	最推荐的默认 Loader；
	•	适合大多数通用文本文档；
	•	对编码与文件结构自动识别，几乎无需参数。



### ② PyPDFLoader —— 结构化 PDF 加载器

🔹 基于 PyPDF2 实现；
🔹 能逐页提取文本内容，并保留页码信息。

✅ 示例代码

In [ ]:
from langchain.document_loaders import PyPDFLoader

loader = PyPDFLoader("../data/sample.pdf")
docs = loader.load()

print(f"文档共 {len(docs)} 页")
print(docs[0].metadata)  # {'source': 'data/sample.pdf', 'page': 0}
print(docs[0].page_content[:200])

📘 使用场景

	•	原生数字 PDF（非扫描版）；
	•	希望保留页码、章节等结构信息；
	•	可结合 TextSplitter 分页拆分长文档。

### ③ CSVLoader —— 表格文本加载器

🔹 用于加载 .csv 文件，将每一行转化为 Document；
🔹 自动识别编码，可指定分隔符、字段映射。

✅ 示例代码

In [ ]:
from langchain.document_loaders import CSVLoader

loader = CSVLoader(file_path="data/finance.csv", encoding="utf-8")
docs = loader.load()

print(f"共加载 {len(docs)} 行数据")
print(docs[0].page_content)



[
 Document(page_content='question: What is LangChain?\nanswer: LangChain is a framework for building LLM-powered applications.\ncategory: AI', metadata={'source': '/Volumes/PSSD/未命名文件夹/donwload/创建知识库数据库/document_loaders/test.csv', 'row': 0}), 
 Document(page_content='question: What is ChatGPT?\nanswer: ChatGPT is an AI developed by OpenAI.\ncategory: AI', metadata={'source': '/Volumes/PSSD/未命名文件夹/donwload/创建知识库数据库/document_loaders/test.csv', 'row': 1}), 
 Document(page_content='question: What is Python?\nanswer: Python is a popular programming language.\ncategory: Programming', metadata={'source': '/Volumes/PSSD/未命名文件夹/donwload/创建知识库数据库/document_loaders/test.csv', 'row': 2})
 ]
 📘 使用场景

	•	表格类知识文件（如财报、指标表）；
	•	结构化数据转换为自然语言语料；
	•	适合与数值分析任务结合。

### ④ JSONLoader —— JSON 文件加载器

🔹 用于解析 .json 文件，可通过 jq_schema 提取特定字段。
🔹 可处理层级结构，支持多级嵌套。

✅ 示例代码

In [ ]:
from langchain.document_loaders import JSONLoader

# jq_schema="." 表示提取整个JSON
loader = JSONLoader(file_path="data/config.json", jq_schema=".", text_content=False)
docs = loader.load()

print(f"共加载 {len(docs)} 条数据")
print(docs[0].page_content)

📘 使用场景

	•	结构化文本（如知识库配置、API响应）；
	•	结合 jq_schema 实现定向提取：

jq_schema=".items[].description"


### ⑤ DirectoryLoader —— 文件夹批量加载器

🔹 一次性加载目录中的所有文件；
🔹 支持指定文件后缀、过滤器。

✅ 示例代码

In [ ]:
from langchain.document_loaders import DirectoryLoader

loader = DirectoryLoader(
    path="data/docs",
    glob="**/*.txt"  # 匹配所有 txt 文件
)
docs = loader.load()

print(f"共加载 {len(docs)} 个文件")
print(docs[0].metadata)

📘 使用场景

	•	批量加载知识库；
	•	自动扫描整个文件夹；
	•	可配合 RecursiveCharacterTextSplitter 拆分后入库。

### 6️⃣ 自定义Imgloader

In [ ]:
from typing import List
from langchain.document_loaders.unstructured import UnstructuredFileLoader
from document_loaders.ocr import get_ocr


class RapidOCRLoader(UnstructuredFileLoader):
    def _get_elements(self) -> List:
        def img2text(filepath):
            resp = ""
            ocr = get_ocr()
            result, _ = ocr(filepath)
            if result:
                ocr_result = [line[1] for line in result]
                resp += "\n".join(ocr_result)
            return resp

        text = img2text(self.file_path)
        from unstructured.partition.text import partition_text
        return partition_text(text=text, **self.unstructured_kwargs)


if __name__ == "__main__":
    loader = RapidOCRLoader(file_path="/Volumes/PSSD/未命名文件夹/donwload/创建知识库数据库/knowledge_base/samples/content/llm/img/大模型技术栈-算法与原理-幕布图片-81470-404273.jpg")
    docs = loader.load()
    print(docs)


### 7️⃣ 自定义PDFloader

In [ ]:
from typing import List
from langchain.document_loaders.unstructured import UnstructuredFileLoader
from document_loaders.ocr import get_ocr
import tqdm


class RapidOCRPDFLoader(UnstructuredFileLoader):
    def _get_elements(self) -> List:
        def pdf2text(filepath):
            import fitz # pyMuPDF里面的fitz包，不要与pip install fitz混淆
            import numpy as np
            ocr = get_ocr()
            doc = fitz.open(filepath)
            resp = ""

            b_unit = tqdm.tqdm(total=doc.page_count, desc="RapidOCRPDFLoader context page index: 0")
            for i, page in enumerate(doc):

                # 更新描述
                b_unit.set_description("RapidOCRPDFLoader context page index: {}".format(i))
                # 立即显示进度条更新结果
                b_unit.refresh()
                # TODO: 依据文本与图片顺序调整处理方式
                text = page.get_text("text")
                resp += text + "\n"

                img_list = page.get_images()
                for img in img_list:
                    pix = fitz.Pixmap(doc, img[0])
                    img_array = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, -1)
                    result, _ = ocr(img_array)
                    if result:
                        ocr_result = [line[1] for line in result]
                        resp += "\n".join(ocr_result)

                # 更新进度
                b_unit.update(1)
            return resp

        text = pdf2text(self.file_path)
        from unstructured.partition.text import partition_text
        return partition_text(text=text, **self.unstructured_kwargs)


if __name__ == "__main__":
    loader = RapidOCRPDFLoader(file_path="/Volumes/PSSD/未命名文件夹/donwload/创建知识库数据库/langchain.pdf")
    docs = loader.load()
    print(docs)


### 💡 实战建议

| 场景 | 推荐 Loader |
|------|--------------|
| 通用文本（TXT / DOCX / HTML / MD） | `UnstructuredFileLoader` |
| PDF 文档（数字版） | `PyPDFLoader` |
| 表格数据 | `CSVLoader` |
| 结构化 JSON | `JSONLoader` |
| 批量加载文件夹 | `DirectoryLoader` |
